# Bone Detection in Video using Hough Transform

This notebook implements a video processing pipeline that counts the number of collected bones (by counting animations for each bone collected) passes through a defined region of interest (ROI) in a video. The approach combines:

- **HSV color masking** to isolate the target object
- **Morphological operations** to clean up the mask
- **Hough Circle Transform** to detect circular shapes
- **Cooldown logic** to avoid double-counting the same detection

## 1. Imports

In [ ]:
import numpy as np
import cv2
import os
import matplotlib.pyplot as plt

## 2. Configuration

### 2.1 Target Color

The circle/ball is identified by its representative RGB color. The HSV tolerance values control how broadly the color mask accepts similar shades.

In [ ]:
circle_rgb = [219, 203, 176]  # Target object color in RGB

h_tol = 5    # Hue tolerance
s_tol = 45   # Saturation tolerance
v_tol = 35   # Value tolerance

### 2.2 Morphological Parameters

These control the mask cleanup pipeline. Erosions thin the mask to remove small noise; dilations expand it to fill gaps; open/close operations handle more structured artifacts.

In [ ]:
k_size    = 5   # Kernel size for morphological ops
erodes    = 0   # Erosion iterations
dilations = 1   # Dilation iterations
openings  = 1   # Morphological opening iterations
closings  = 2   # Morphological closing iterations

# Frames to wait after a detection before counting again
cooldown_frames = 7

### 2.3 Hough Circle Parameters

These parameters are passed to `cv2.HoughCircles`. References:
- [OpenCV HoughCircles docs](https://docs.opencv.org/4.12.0/dd/d1a/group__imgproc__feature.html#ga47849c3be0d0406ad3ca45db65a25d2d)
- [Hough Circle tutorial](https://docs.opencv.org/4.x/d4/d70/tutorial_hough_circle.html)

In [ ]:
dp        = 1    # Inverse ratio of accumulator resolution to image resolution
minDist   = 50   # Minimum distance between detected circle centers
p1        = 45   # Upper threshold for Canny edge detector
p2        = 15   # Accumulator threshold for circle detection
minRadius = 40   # Minimum circle radius (pixels)
maxRadius = 85   # Maximum circle radius (pixels)

## 3. HSV Range Helper

Converts a reference RGB color into an HSV detection range by expanding ± tolerance in each channel.

In [ ]:
def make_hsv_range(rgb, h_tol, s_tol, v_tol):
    """
    Convert an RGB color to HSV and build a detection range
    by expanding ± tolerance in each channel.

    Parameters
    ----------
    rgb   : list[int]  Reference color as [R, G, B]
    h_tol : int        Hue tolerance (0–179 in OpenCV)
    s_tol : int        Saturation tolerance (0–255)
    v_tol : int        Value tolerance (0–255)

    Returns
    -------
    lower, upper : np.ndarray  HSV lower and upper bounds
    """
    hsv = cv2.cvtColor(np.uint8([[rgb]]), cv2.COLOR_RGB2HSV)[0][0]
    hsv = hsv.astype(int)

    lower = np.array([
        max(hsv[0] - h_tol, 0),
        max(hsv[1] - s_tol, 0),
        max(hsv[2] - v_tol, 0)
    ], dtype=np.uint8)

    upper = np.array([
        min(hsv[0] + h_tol, 179),
        min(hsv[1] + s_tol, 255),
        min(hsv[2] + v_tol, 255)
    ], dtype=np.uint8)

    return lower, upper


lower, upper = make_hsv_range(circle_rgb, h_tol, s_tol, v_tol)
print(f"HSV lower: {lower}")
print(f"HSV upper: {upper}")

## 4. Video Processing Pipeline

### Pipeline Steps per Frame

1. **Crop ROI** — focus on the bottom-center of the frame where the object is expected
2. **HSV masking** — isolate pixels matching the target color
3. **Morphological cleanup** — erode, dilate, close, open
4. **Gaussian blur** — smooth edges for Hough Transform
5. **Hough Circle detection** — find circular blobs
6. **Cooldown counter** — suppress duplicate counts for the same passing event

### Region of Interest (ROI)

```
┌─────────────────────────────┐
│        (ignored)            │  0% – 50% height
├──────────┬──────┬───────────┤
│ ignored  │ ROI  │  ignored  │  50%–100% height, 35%–65% width
└──────────┴──────┴───────────┘
```

In [ ]:
def process_video(video_path, debug=False, start_frame=0):
    """
    Count circle detections in a video using Hough Circle Transform.

    Parameters
    ----------
    video_path  : str   Path to the input video file
    debug       : bool  If True, display annotated frames inline (Jupyter-friendly)
    start_frame : int   Frame index to start processing from

    Returns
    -------
    int  Total number of detected circle events
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"ERROR: Could not open video: {video_path}")
        return 0

    if start_frame > 0:
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    count    = 0
    cooldown = 0
    kernel   = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_size, k_size))
    lower_v, upper_v = make_hsv_range(circle_rgb, h_tol, s_tol, v_tol)

    while True:
        current_frame_idx = int(cap.get(cv2.CAP_PROP_POS_FRAMES))
        grabbed, frame = cap.read()
        if not grabbed:
            break

        # --- ROI crop ---
        height, width = frame.shape[:2]
        roi_y1, roi_y2 = int(height * 0.5), int(height * 1.0)
        roi_x1, roi_x2 = int(width  * 0.35), int(width  * 0.65)
        roi = frame[roi_y1:roi_y2, roi_x1:roi_x2]

        # --- HSV masking ---
        hsv_roi   = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
        mask_raw  = cv2.inRange(hsv_roi, lower_v, upper_v)

        # --- Morphological cleanup ---
        mask_clean = cv2.erode(mask_raw,   kernel, iterations=erodes)
        mask_clean = cv2.dilate(mask_clean, kernel, iterations=dilations)
        mask_clean = cv2.morphologyEx(mask_clean, cv2.MORPH_CLOSE, kernel, iterations=closings)
        mask_clean = cv2.morphologyEx(mask_clean, cv2.MORPH_OPEN,  kernel, iterations=openings)

        # --- Blur before Hough ---
        blurred = cv2.GaussianBlur(mask_clean, (5, 5), 0)

        # --- Hough Circle Transform ---
        # Reference: https://docs.opencv.org/4.x/d4/d70/tutorial_hough_circle.html
        circles = cv2.HoughCircles(
            blurred, cv2.HOUGH_GRADIENT,
            dp=dp, minDist=minDist,
            param1=p1, param2=p2,
            minRadius=minRadius, maxRadius=maxRadius
        )

        detected = circles is not None

        if detected and cooldown == 0:
            count   += 1
            cooldown = cooldown_frames

        if cooldown > 0:
            cooldown -= 1

        # --- Jupyter-friendly debug display (LLM-assisted) ---
        if debug:
            debug_frame = cv2.cvtColor(roi.copy(), cv2.COLOR_BGR2RGB)
            if circles is not None:
                for i in np.uint16(np.around(circles))[0]:
                    cv2.circle(debug_frame, (i[0], i[1]), i[2], (0, 255, 0), 2)

            border_color = (255, 0, 0) if cooldown > 0 else (200, 200, 200)
            cv2.rectangle(debug_frame, (0, 0),
                          (debug_frame.shape[1]-1, debug_frame.shape[0]-1),
                          border_color, 4)
            cv2.putText(debug_frame,
                        f"Frame: {current_frame_idx}  Count: {count}",
                        (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

            fig, axes = plt.subplots(1, 2, figsize=(12, 4))
            axes[0].imshow(debug_frame)
            axes[0].set_title(f'ROI — Frame {current_frame_idx} | Detections: {count}')
            axes[0].axis('off')
            axes[1].imshow(mask_clean, cmap='gray')
            axes[1].set_title('Cleaned Mask')
            axes[1].axis('off')
            plt.tight_layout()
            plt.show()

    cap.release()
    return count

## 5. Evaluation

### 5.1 Load Ground Truth

The ground truth CSV (`count.csv`) contains columns `video_name` and `true_count`.

In [ ]:
DATA_DIR = "data"

ground_truth = {}
with open(os.path.join(DATA_DIR, 'count.csv'), 'r') as f:
    lines = f.readlines()[1:]  # skip header

for line in lines:
    video_name, true_count = line.strip().split(',')
    ground_truth[video_name] = int(true_count)

print(f"Loaded ground truth for {len(ground_truth)} videos.")
for k, v in ground_truth.items():
    print(f"  {k}: {v}")

### 5.2 Run Predictions

In [ ]:
results = {}

for video_name, true_count in ground_truth.items():
    video_path = os.path.join(DATA_DIR, f"{video_name}.mp4")
    pred_count = process_video(video_path, debug=False, start_frame=0)
    results[video_name] = pred_count
    print(f"{video_name:<30} true={true_count:>3}  pred={pred_count:>3}  error={abs(true_count - pred_count):>3}")

print("\nPredictions complete.")

### 5.3 Compute Mean Absolute Error (MAE)

In [ ]:
mae = sum(abs(ground_truth[k] - results[k]) for k in ground_truth) / len(ground_truth)
print(f"Mean Absolute Error (MAE): {mae:.2f}")

### 5.4 Results Table

In [ ]:
print(f"{'Video':<30} {'True':>6} {'Pred':>6} {'Error':>6}")
print("-" * 52)
for name in sorted(ground_truth.keys()):
    true = ground_truth[name]
    pred = results.get(name, 0)
    err  = abs(true - pred)
    print(f"{name:<30} {true:>6} {pred:>6} {err:>6}")
print("-" * 52)
print(f"{'MAE':<30} {'':>6} {'':>6} {mae:>6.2f}")

## 6. Visualisation

### 6.1 Prediction vs Ground Truth

In [ ]:
names  = sorted(ground_truth.keys())
trues  = [ground_truth[n] for n in names]
preds  = [results.get(n, 0) for n in names]

x = range(len(names))
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar([i - 0.2 for i in x], trues, width=0.4, label='Ground Truth', color='steelblue')
ax.bar([i + 0.2 for i in x], preds, width=0.4, label='Predicted',    color='coral')
ax.set_xticks(list(x))
ax.set_xticklabels(names, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Circle Count')
ax.set_title(f'Ground Truth vs Predicted  (MAE = {mae:.2f})')
ax.legend()
plt.tight_layout()
plt.show()

### 6.2 Debug — Inspect a Single Video Frame by Frame

Set `DEBUG_VIDEO` to any video name and run the cell to visualise the mask and detections inline.

In [ ]:
DEBUG_VIDEO   = list(ground_truth.keys())[0]  # Change to any video name
DEBUG_START   = 0                              # Starting frame index

debug_path = os.path.join(DATA_DIR, f"{DEBUG_VIDEO}.mp4")
print(f"Debugging: {DEBUG_VIDEO}")
result = process_video(debug_path, debug=True, start_frame=DEBUG_START)
print(f"Total detected: {result}  |  Ground truth: {ground_truth[DEBUG_VIDEO]}")